# Context-Aware Chatbot Using LangChain + RAG

**Task:** Build a chatbot that (1) remembers earlier turns in the conversation and (2) pulls in relevant facts from a vectorized knowledge base before answering, instead of relying purely on the LLM's own memorized knowledge.

**Approach:**
- **Knowledge base**: a small custom corpus built from a handful of Wikipedia pages (easy to swap for your own PDFs/txt files/internal docs - see the note in section 2)
- **Embeddings**: `sentence-transformers/all-MiniLM-L6-v2` (small, fast, works fine on CPU)
- **Vector store**: FAISS
- **LLM**: `google/flan-t5-base` run locally via a HuggingFace pipeline, wrapped as a LangChain LLM - this avoids needing an OpenAI API key, which felt more appropriate for an assignment notebook
- **Memory**: LangChain's `ConversationBufferMemory` so follow-up questions ("what about its capital?") resolve correctly
- **Chain**: `ConversationalRetrievalChain`, which combines retrieval + memory in one call

Everything runs locally/offline after the initial Wikipedia fetch, so this should work fine on Kaggle with internet access turned on (Settings -> Internet -> On), which is needed once, just to pull the source pages.


In [1]:
!pip install -q langchain langchain-community langchain-huggingface faiss-cpu sentence-transformers wikipedia transformers accelerate

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 28.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 75.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 558.3/558.3 kB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which i

## 1. Imports

In [2]:
import wikipedia

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_classic.memory import ConversationBufferMemory
from langchain_classic.chains import ConversationalRetrievalChain
from langchain_core.documents import Document

from transformers import pipeline


/tmp/ipykernel_58/4153359983.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


## 2. Build the knowledge base

For this notebook I pulled a handful of Wikipedia pages as a stand-in "custom corpus" - swap `TOPICS` for whatever your actual knowledge base should cover, or replace this whole cell with a loop that reads `.txt`/`.pdf` files from a folder instead if you're using internal documents.

In [5]:
import wikipedia
from langchain_text_splitters import RecursiveCharacterTextSplitter  # Modern splitter path

TOPICS = [
    "Artificial intelligence",
    "Machine learning",
    "Natural language processing",
    "Large language model",
    "Retrieval-augmented generation",
]

raw_docs = []
for topic in TOPICS:
    try:
        # Use user_agent to comply with Wikipedia policy and prevent blocking
        wikipedia.set_user_agent("MyLangChainBot/1.0 (contact@example.com)")
        page = wikipedia.page(topic, auto_suggest=False)
        raw_docs.append(Document(page_content=page.content, metadata={"source": page.title}))
        print(f"✅ Fetched: {page.title} ({len(page.content)} chars)")
    except Exception as e:
        print(f"❌ Skipped '{topic}': {e}")

✅ Fetched: Artificial intelligence (85281 chars)
✅ Fetched: Machine learning (58709 chars)
✅ Fetched: Natural language processing (32382 chars)
✅ Fetched: Large language model (56502 chars)
✅ Fetched: Retrieval-augmented generation (10903 chars)


In [3]:
TOPICS = [
    "Artificial intelligence",
    "Machine learning",
    "Natural language processing",
    "Large language model",
    "Retrieval-augmented generation",
]

raw_docs = []
for topic in TOPICS:
    try:
        page = wikipedia.page(topic, auto_suggest=False)
        raw_docs.append(Document(page_content=page.content, metadata={"source": page.title}))
        print(f"fetched: {page.title} ({len(page.content)} chars)")
    except Exception as e:
        print(f"skipped {topic}: {e}")


skipped Artificial intelligence: Expecting value: line 1 column 1 (char 0)
skipped Machine learning: Expecting value: line 1 column 1 (char 0)
skipped Natural language processing: Expecting value: line 1 column 1 (char 0)
skipped Large language model: Expecting value: line 1 column 1 (char 0)
skipped Retrieval-augmented generation: Expecting value: line 1 column 1 (char 0)


**If you're using your own files instead of Wikipedia**, this is roughly what that cell would look like:

```python
import os
from langchain.docstore.document import Document

folder = "my_docs"
raw_docs = []
for fname in os.listdir(folder):
    with open(os.path.join(folder, fname), "r", encoding="utf-8") as f:
        raw_docs.append(Document(page_content=f.read(), metadata={"source": fname}))
```

Everything downstream (chunking, embedding, retrieval) stays exactly the same either way - that's really the point of RAG, the pipeline doesn't care where the text came from.

## 3. Chunk the documents

Wikipedia pages are long, and stuffing an entire article into the retriever as one chunk would hurt retrieval precision (and blow past the LLM's context window). Splitting into ~500-character chunks with a bit of overlap so context doesn't get awkwardly cut off mid-sentence at chunk boundaries.

In [6]:
if len(raw_docs) == 0:
    print("\n⚠️ Error: No documents were successfully fetched! Check your internet or package setup.")
else:
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=50,
        separators=["\n\n", "\n", ". ", " ", ""],
    )

    chunks = text_splitter.split_documents(raw_docs)
    print(f"\n🎉 Success: {len(raw_docs)} documents processed into {len(chunks)} chunks.")
    print(f"--- Sample from first chunk ---\n{chunks[0].page_content[:300]}")



🎉 Success: 5 documents processed into 816 chunks.
--- Sample from first chunk ---
Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics and computer science that develo


## 4. Embed the chunks and build the FAISS index

`all-MiniLM-L6-v2` is a good default for this kind of task - it's a fraction of the size of larger embedding models but still gives solid semantic search quality, and it runs comfortably on CPU so this isn't GPU-dependent.

In [7]:
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vectorstore = FAISS.from_documents(chunks, embedding_model)

# save it so we don't have to re-embed everything every time we restart the kernel (or for the Streamlit app later)
vectorstore.save_local("faiss_index")
print("vector store built and saved")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

vector store built and saved


In [8]:
# quick sanity check - does retrieval actually pull relevant chunks?
query = "What is retrieval-augmented generation?"
results = vectorstore.similarity_search(query, k=3)
for i, doc in enumerate(results):
    print(f"--- result {i+1} (source: {doc.metadata['source']}) ---")
    print(doc.page_content[:250], "\n")


--- result 1 (source: Large language model) ---
=== Retrieval-augmented generation === 

--- result 2 (source: Retrieval-augmented generation) ---
Retrieval-augmented generation (RAG) enhances large language models (LLMs) by incorporating an information-retrieval mechanism that allows models to access and utilize additional data beyond their original training set. Ars Technica notes that "when  

--- result 3 (source: Large language model) ---
Retrieval-augmented generation (RAG) is an approach that integrates LLMs with document retrieval systems. Given a query, a document retriever is called to retrieve the most relevant documents. This is usually done by encoding the query and the docume 



## 5. Set up the local LLM

Using `flan-t5-base` here since it's instruction-tuned, small enough to run without a beefy GPU, and doesn't need an API key. It's not going to write essays, but it's decent at focused QA-style answers, which is exactly what a RAG chatbot needs it for. If you have access to a larger model (e.g. Llama or Mistral via `HuggingFaceHub`, or OpenAI's API), you can drop it in here instead - the rest of the chain doesn't change.

In [10]:
hf_pipeline = pipeline(
    "text-generation",
    model="google/flan-t5-base",
    max_length=256,
    temperature=0.3,
)

llm = HuggingFacePipeline(pipeline=hf_pipeline)


model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Passing `generation_config` together with generation-related arguments=({'temperature', 'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 

## 6. Conversational memory + the RAG chain

`ConversationBufferMemory` keeps the running chat history and feeds it back into the chain each turn, which is what lets the bot resolve things like "it" or "what about..." in follow-up questions. `ConversationalRetrievalChain` wires that memory together with the FAISS retriever: on every turn it (1) rewrites the question using chat history if needed, (2) retrieves relevant chunks, (3) asks the LLM to answer using those chunks as context.

In [11]:
memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True,
    output_key="answer",
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

qa_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    memory=memory,
    return_source_documents=True,
)


/tmp/ipykernel_58/1806376473.py:1: LangChainDeprecationWarning: The class `ConversationBufferMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = ConversationBufferMemory(


## 7. Try it out - does memory + retrieval actually work?

The second question deliberately doesn't repeat "large language models" by name, to check that the memory is actually letting the chain resolve the follow-up correctly instead of just answering the literal words in isolation.

In [12]:
def ask(question):
    result = qa_chain.invoke({"question": question})
    print("Q:", question)
    print("A:", result["answer"])
    print("sources:", [doc.metadata["source"] for doc in result["source_documents"]])
    print()
    return result


_ = ask("What is a large language model?")
_ = ask("What are they commonly used for?")
_ = ask("How does retrieval-augmented generation improve them?")


Q: What is a large language model?
A: Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

. The qualifier "large" in "large language model" is inherently vague, as there is no definitive threshold for the number of parameters required to qualify as "large".

== Limitations and challenges ==
Despite sophisticated architectures and massive scale, large language models exhibit persistent and well-documented limitations that constrain their deployment in high-stakes applications.


=== Hallucinations ===

A large language model (LLM) is a neural network trained on a vast amount of text for natural language processing tasks, especially language generation. LLMs can typically generate, summarize, translate, and analyze text in many contexts, and are a foundational technology behind modern chatbots. Biased or inaccurate training data can make an LLM's output less reliable.

Question

Token indices sequence length is longer than the specified maximum sequence length for this model (806 > 512). Running this sequence through the model will result in indexing errors


Q: What are they commonly used for?
A: Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

A large language model (LLM) is a neural network trained on a vast amount of text for natural language processing tasks, especially language generation. LLMs can typically generate, summarize, translate, and analyze text in many contexts, and are a foundational technology behind modern chatbots. Biased or inaccurate training data can make an LLM's output less reliable.

. The qualifier "large" in "large language model" is inherently vague, as there is no definitive threshold for the number of parameters required to qualify as "large".

== Limitations and challenges ==
Despite sophisticated architectures and massive scale, large language models exhibit persistent and well-documented limitations that constrain their deployment in high-stakes applications.


=== Hallucinations ===

Questio

If memory is working correctly, the second question's answer should clearly be about LLM use cases (not, say, a generic answer that ignores the first question entirely), and the third should connect RAG back to the LLM context from earlier turns.

## 8. Streamlit app for live interaction

Same as before - Kaggle can't host a live server, so this writes `app.py` to run locally. It reloads the saved FAISS index rather than re-embedding everything from scratch, and keeps chat history in Streamlit's `session_state` so memory persists across turns within a browser session.

In [14]:
%%writefile app.py
import streamlit as st

from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationalRetrievalChain
from transformers import pipeline

st.set_page_config(page_title="RAG Chatbot", page_icon="\U0001F4DA")
st.title("Context-Aware RAG Chatbot")


@st.cache_resource
def load_chain():
    embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    vectorstore = FAISS.load_local("faiss_index", embedding_model, allow_dangerous_deserialization=True)

    hf_pipeline = pipeline("text2text-generation", model="google/flan-t5-base", max_length=256, temperature=0.3)
    llm = HuggingFacePipeline(pipeline=hf_pipeline)

    memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True, output_key="answer")
    retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

    chain = ConversationalRetrievalChain.from_llm(
        llm=llm, retriever=retriever, memory=memory, return_source_documents=True
    )
    return chain


chain = load_chain()

if "messages" not in st.session_state:
    st.session_state.messages = []

for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.write(msg["content"])

user_input = st.chat_input("Ask something about the knowledge base...")

if user_input:
    st.session_state.messages.append({"role": "user", "content": user_input})
    with st.chat_message("user"):
        st.write(user_input)

    with st.chat_message("assistant"):
        with st.spinner("Thinking..."):
            result = chain.invoke({"question": user_input})
            answer = result["answer"]
            sources = sorted(set(doc.metadata["source"] for doc in result["source_documents"]))

            st.write(answer)
            if sources:
                st.caption("Sources: " + ", ".join(sources))

    st.session_state.messages.append({"role": "assistant", "content": answer})


Overwriting app.py


### Running it

Make sure the `faiss_index/` folder from section 4 is sitting next to `app.py`, then:

```bash
streamlit run app.py
```

This opens a normal chat UI in the browser - type a question, get an answer grounded in the retrieved chunks, and the source Wikipedia page(s) are shown under each answer so you can sanity-check the retrieval.

### Notes / things I'd improve with more time
- `flan-t5-base` is small and sometimes gives fairly terse or overly literal answers - swapping in a bigger instruction-tuned model (Mistral-7B-Instruct, Llama-3-8B-Instruct, etc. via `HuggingFaceHub` or a local `transformers` pipeline with 4-bit quantization) would give noticeably better conversational quality if GPU memory allows
- Right now chunk size/overlap (500/50) is just a reasonable default - didn't tune it against a real eval set of question/answer pairs, which would be the proper way to pick these numbers
- Could add re-ranking (e.g. a cross-encoder) on top of the initial FAISS similarity search to improve retrieval precision when the corpus gets larger than a handful of pages
